# requires-grad-leaf-assert — worked example 2: Audit All Parameters and Collect All Bad Ones Before Raising

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `requires-grad-leaf-assert`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Raising on the first bad parameter is fine for a quick guard, but a better debugging tool collects all problematic tensors and reports them together. This avoids the frustrating cycle of fix-one, re-run, find-next. The audit function iterates all parameters, checks both `is_leaf` and `requires_grad`, and returns a structured report of every failure.

## Worked solution

**Step 1 — iterate with `enumerate`.** Using `enumerate(params)` gives the index `i` alongside the tensor, which is essential for locating the problem.

**Step 2 — check both conditions.** For each tensor, check `is_leaf` and `requires_grad` independently.

**Step 3 — determine the category.** If the tensor is non-leaf, that is the more fundamental issue (even fixing `requires_grad` wouldn't help). If it is a leaf but lacks `requires_grad`, it's a no-grad case.

**Step 4 — collect results.** Append a dict with `{index, shape, is_leaf, requires_grad, category}` for each bad tensor. Continue past failures (don't break).

**Step 5 — return the list.** The caller decides whether to raise or log. Empty list means all params are safe.

In [ ]:
import torch as t
import torch.nn as nn

def audit_params(params) -> list:
    """Return a list of dicts describing every optimizer-unsafe param."""
    problems = []
    for i, p in enumerate(params):
        if p.is_leaf and p.requires_grad:
            continue  # this one is fine
        category = 'non-leaf' if not p.is_leaf else 'no-grad'
        problems.append({
            'index': i,
            'shape': tuple(p.shape),
            'is_leaf': bool(p.is_leaf),
            'requires_grad': bool(p.requires_grad),
            'category': category,
        })
    return problems

# --- exercise and print ---
t.manual_seed(0)

good  = nn.Parameter(t.randn(2, 3))       # leaf + requires_grad
noleaf_src = t.randn(4, requires_grad=True)
non_leaf = noleaf_src + 1                  # non-leaf
no_grad  = t.randn(5)                      # leaf, no grad

params = [good, non_leaf, no_grad, nn.Parameter(t.zeros(1))]
problems = audit_params(params)

print(f'Found {len(problems)} problem(s):')
for p in problems:
    print(f'  index={p["index"]} shape={p["shape"]} category={p["category"]} '
          f'is_leaf={p["is_leaf"]} requires_grad={p["requires_grad"]}')